# LeetCode 2389 — Longest Subsequence With Limited Sum bằng Python

Link đề: https://leetcode.com/problems/longest-subsequence-with-limited-sum/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting, Prefix Sum, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def answerQueries(self, nums: List[int], queries: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu "dãy con" (subsequence) và vì sao bài này **không cần giữ thứ tự** mà vẫn đúng.
2. Nhận ra chiến lược **tham lam (greedy)**: muốn chọn nhiều phần tử nhất với tổng không vượt `q`, hãy chọn các số **nhỏ nhất** trước.
3. Dùng **sắp xếp + prefix sum + tìm kiếm nhị phân** (`bisect_right`) để trả lời mỗi truy vấn trong `O(log n)`.
4. Viết 4 cách: mô phỏng greedy cho từng truy vấn, quét tuyến tính prefix sum, `bisect`, và hai con trỏ offline (sắp xếp truy vấn); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: quên sắp xếp, dùng `bisect_left` thay `bisect_right`, trả về chỉ số thay vì độ dài, sửa/ghi đè mất thứ tự truy vấn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n` và mảng `queries` độ dài `m`. Với **mỗi** `queries[i]`, hãy tìm **độ dài lớn nhất** của một **dãy con** của `nums` sao cho **tổng các phần tử ≤ `queries[i]`**. Trả về mảng `answer` độ dài `m` với `answer[i]` là đáp án của truy vấn `i`.

*Dãy con* là dãy thu được bằng cách xóa một số (có thể không) phần tử mà **không đổi thứ tự** các phần tử còn lại.

| Ví dụ | nums | queries | Output |
|---|---|---|---|
| 1 | `[4,5,2,1]` | `[3,10,21]` | `[2,3,4]` |
| 2 | `[2,3,4,5]` | `[1]` | `[0]` |

Giải thích ví dụ 1: với `q = 3` chọn `[2,1]` (tổng 3) → độ dài 2; với `q = 10` chọn `[4,2,1]` (tổng 7) → 3; với `q = 21` lấy cả 4 số (tổng 12) → 4.

Constraints: `n == nums.length`, `m == queries.length`; `1 <= n, m <= 1000`; `1 <= nums[i] <= 10^6`; `1 <= queries[i] <= 10^6`.

In [1]:
nums, queries = [4, 5, 2, 1], [3, 10, 21]
print("tổng cả mảng:", sum(nums))
for q in queries:
    print(f"q={q}: tổng cả mảng {'<=' if sum(nums) <= q else '>'} q")

tổng cả mảng: 12
q=3: tổng cả mảng > q
q=10: tổng cả mảng > q
q=21: tổng cả mảng <= q


## 3. Kiến thức nền

### 3.1. Sắp xếp `sorted` và `list.sort`

`sorted(a)` trả về list mới đã sắp xếp tăng dần; `a.sort()` sắp xếp tại chỗ và trả `None`. Độ phức tạp `O(n log n)`.

In [2]:
a = [4, 5, 2, 1]
b = sorted(a)
print(a, b)
a.sort()
print(a, "<- sort() đổi chính a")
print("sai thường gặp:", [3, 1].sort())

[4, 5, 2, 1] [1, 2, 4, 5]
[1, 2, 4, 5] <- sort() đổi chính a
sai thường gặp: None


### 3.2. Prefix sum với `itertools.accumulate`

`pre[i]` = tổng `i + 1` phần tử đầu của mảng đã sắp xếp. Python có `itertools.accumulate` để tính nhanh; hoặc tự cộng dồn bằng vòng `for`. Ở bài này, `pre[j]` chính là **tổng nhỏ nhất có thể đạt được khi chọn `j + 1` phần tử**.

In [3]:
from itertools import accumulate
nums = sorted([4, 5, 2, 1])
pre = list(accumulate(nums))
print(nums, pre)
manual, run = [], 0
for x in nums:
    run += x
    manual.append(run)
print(manual == pre)

[1, 2, 4, 5] [1, 3, 7, 12]
True


### 3.3. Tìm kiếm nhị phân: `bisect_left` và `bisect_right`

Trên mảng **đã sắp xếp** `p`:

- `bisect_left(p, x)`: vị trí chèn `x` sao cho nằm **trước** các giá trị bằng `x`; = số phần tử `< x`.
- `bisect_right(p, x)`: vị trí chèn `x` nằm **sau** các giá trị bằng `x`; = số phần tử `<= x`.

Bài này cần "đếm số tiền tố có tổng `<= q`" nên dùng **`bisect_right`**.

In [4]:
from bisect import bisect_left, bisect_right
p = [1, 3, 7, 12]
for x in (0, 1, 3, 5, 12, 20):
    print(f"x={x:2d}: bisect_left={bisect_left(p, x)}, bisect_right={bisect_right(p, x)}")

x= 0: bisect_left=0, bisect_right=0
x= 1: bisect_left=0, bisect_right=1
x= 3: bisect_left=1, bisect_right=2
x= 5: bisect_left=2, bisect_right=2
x=12: bisect_left=3, bisect_right=4
x=20: bisect_left=4, bisect_right=4


### 3.4. Hai con trỏ trên hai dãy đã sắp xếp (offline)

Nếu cả `pre` và các truy vấn đều **tăng dần**, con trỏ `j` trên `pre` chỉ cần đi **một chiều**. Phải nhớ vị trí gốc của mỗi truy vấn để trả đáp án đúng thứ tự — dùng `sorted(range(m), key=lambda i: queries[i])`.

In [5]:
queries = [10, 3, 21]
order = sorted(range(len(queries)), key=lambda i: queries[i])
print("thứ tự chỉ số theo giá trị tăng dần:", order, [queries[i] for i in order])

thứ tự chỉ số theo giá trị tăng dần: [1, 0, 2] [3, 10, 21]


### 3.5. Tạo list kết quả đúng kích thước

`ans = [0] * m` tạo `m` số 0 (an toàn vì số nguyên bất biến), rồi gán `ans[i] = ...` theo chỉ số gốc.

In [6]:
m = 3
ans = [0] * m
ans[1] = 5
print(ans)

[0, 5, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giả sử bạn có `q` đồng tiền và muốn mua **càng nhiều món** càng tốt, mỗi món có giá `nums[i]`. Bạn sẽ mua món **rẻ nhất** trước, rồi món rẻ tiếp theo... cho đến khi hết tiền. Thứ tự các món ban đầu **không liên quan** — vì ta chỉ cần *độ dài* của dãy con, mà với mọi tập chỉ số ta đều có thể xếp thành dãy con theo thứ tự chỉ số gốc.

Ví dụ `nums = [4,5,2,1]` → sắp xếp `[1,2,4,5]`, cộng dồn `[1,3,7,12]`:

```
q = 3  : 1 ✓(1), 2 ✓(3), 4 ✗(7>3)         -> 2 món
q = 10 : 1,2,4 ✓ (7), 5 ✗ (12>10)          -> 3 món
q = 21 : tất cả (12 <= 21)                  -> 4 món
```

Chốt: **đáp án = số tiền tố của mảng đã sắp xếp có tổng `<= q`**.

## 5. Cách 1 — Brute Force / baseline

Với mỗi truy vấn: sắp xếp `nums` (một lần), rồi **duyệt từng số nhỏ → lớn**, cộng vào tổng đang có nếu còn vừa, ngược lại dừng. Đếm số lần cộng.

Đây là greedy "mô phỏng" thuần tuý, dễ hiểu nhất. Nút thắt: mỗi truy vấn mất `O(n)` mà ta cộng dồn lại từ đầu, dù tiền tố được tính trước một lần là đủ.

In [7]:
class SolutionBrute:
    def answerQueries(self, nums, queries):
        nums = sorted(nums)
        answer = []
        for q in queries:
            total = 0
            count = 0
            for x in nums:
                if total + x <= q:
                    total += x
                    count += 1
                else:
                    break
            answer.append(count)
        return answer

print(SolutionBrute().answerQueries([4, 5, 2, 1], [3, 10, 21]))
print(SolutionBrute().answerQueries([2, 3, 4, 5], [1]))

[2, 3, 4]
[0]


**Độ phức tạp:** sắp xếp `O(n log n)`; mỗi truy vấn `O(n)` → tổng `O(n log n + m*n)`. Với `n, m <= 1000` là `~10^6` thao tác, vẫn nhanh. Nút thắt: lặp lại phép cộng dồn cho **mỗi truy vấn**. Space `O(n)` (bản sao đã sắp xếp) + `O(m)` kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

Phép cộng dồn không phụ thuộc truy vấn. Tính **một lần** `pre[j] = nums_sorted[0] + ... + nums_sorted[j]`. Mảng `pre` **tăng ngặt** (vì `nums[i] >= 1`). Với mỗi `q`:

- Nếu `pre[j] <= q` thì chọn được `j + 1` phần tử.
- Do `pre` tăng, các `j` thoả mãn là một **tiền tố liên tục**; số phần tử của tiền tố đó = `bisect_right(pre, q)`.

**Quy tắc "nếu… thì…":**

- Nếu `q < nums_sorted[0]` → không chọn được gì → `0`.
- Nếu `q >= pre[-1]` → chọn tất cả → `n`.
- Còn lại → `bisect_right(pre, q)` cho trực tiếp độ dài.

Invariant: `pre` luôn là mảng tăng nên nhị phân hợp lệ; `bisect_right` đếm **chính xác** các tổng `<= q`.

In [8]:
from bisect import bisect_right
from itertools import accumulate
nums, queries = [4, 5, 2, 1], [3, 10, 21]
pre = list(accumulate(sorted(nums)))
print("pre =", pre)
print([bisect_right(pre, q) for q in queries])

pre = [1, 3, 7, 12]
[2, 3, 4]


## 7. Các cách tiếp theo

Ta có 4 cách: (1) brute mô phỏng, (2) prefix sum + quét tuyến tính, (3) prefix sum + `bisect_right`, (4) hai con trỏ offline.

### Cách 2 — Prefix sum + quét tuyến tính

Dựng `pre` một lần, với mỗi truy vấn quét `pre` từ trái, đếm số phần tử `<= q` (dừng khi vượt). Không còn cộng dồn lặp, nhưng vẫn `O(n)` mỗi truy vấn.

In [9]:
class SolutionScan:
    def answerQueries(self, nums, queries):
        pre = []
        run = 0
        for x in sorted(nums):
            run += x
            pre.append(run)
        answer = []
        for q in queries:
            cnt = 0
            while cnt < len(pre) and pre[cnt] <= q:
                cnt += 1
            answer.append(cnt)
        return answer

print(SolutionScan().answerQueries([4, 5, 2, 1], [3, 10, 21]))

[2, 3, 4]


Độ phức tạp Cách 2: Time `O(n log n + m*n)`, Space `O(n)`.

### Cách 3 — Prefix sum + `bisect_right` (cách chuẩn)

Mỗi truy vấn chỉ tốn `O(log n)`. Tổng: `O(n log n + m log n)`.

In [10]:
from bisect import bisect_right
from itertools import accumulate

class SolutionBisect:
    def answerQueries(self, nums, queries):
        pre = list(accumulate(sorted(nums)))
        return [bisect_right(pre, q) for q in queries]

print(SolutionBisect().answerQueries([4, 5, 2, 1], [3, 10, 21]))
print(SolutionBisect().answerQueries([2, 3, 4, 5], [1]))

[2, 3, 4]
[0]


Độ phức tạp Cách 3: Time `O(n log n + m log n)`, Space `O(n)`.

### Cách 4 — Hai con trỏ offline

Sắp xếp **chỉ số** truy vấn theo giá trị tăng dần. Con trỏ `j` (số phần tử đã chọn) chỉ tiến lên. Với truy vấn kế tiếp lớn hơn, ta tiếp tục từ `j` cũ thay vì bắt đầu lại. Tổng `O(n + m)` sau khi sắp xếp (`O(n log n + m log m)`).

In [11]:
class SolutionOffline:
    def answerQueries(self, nums, queries):
        nums = sorted(nums)
        n, m = len(nums), len(queries)
        answer = [0] * m
        j, total = 0, 0                     # j: đã lấy j phần tử, tổng total
        for i in sorted(range(m), key=lambda t: queries[t]):
            q = queries[i]
            while j < n and total + nums[j] <= q:
                total += nums[j]
                j += 1
            answer[i] = j
        return answer

print(SolutionOffline().answerQueries([4, 5, 2, 1], [3, 10, 21]))
print(SolutionOffline().answerQueries([4, 5, 2, 1], [21, 3, 10]))

[2, 3, 4]
[4, 2, 3]


Độ phức tạp Cách 4: Time `O(n log n + m log m)`, Space `O(n + m)`. Cách này là "mẫu" của kỹ thuật **xử lý offline**: đổi thứ tự trả lời để mỗi con trỏ chỉ đi một chiều.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [4,5,2,1]`, `queries = [3,10,21]`: sau sắp xếp `[1,2,4,5]`, `pre = [1,3,7,12]`.

| q | các `pre[j] <= q` | `bisect_right(pre, q)` |
|---|---|---|
| 3 | `1, 3` | 2 |
| 10 | `1, 3, 7` | 3 |
| 21 | `1, 3, 7, 12` | 4 |

In [12]:
def trace_bisect(nums, queries):
    from itertools import accumulate
    s = sorted(nums)
    pre = list(accumulate(s))
    print("sắp xếp:", s, "| pre:", pre)
    out = []
    for q in queries:
        lo, hi = 0, len(pre)            # tự cài bisect_right để thấy từng bước
        while lo < hi:
            mid = (lo + hi) // 2
            print(f"  q={q}: lo={lo} hi={hi} mid={mid} pre[mid]={pre[mid]}", end=" ")
            if pre[mid] <= q:
                lo = mid + 1
                print("-> pre[mid] <= q, lo = mid+1")
            else:
                hi = mid
                print("-> pre[mid] > q, hi = mid")
        print(f"q={q} -> trả lời {lo}")
        out.append(lo)
    return out

print(trace_bisect([4, 5, 2, 1], [3, 10, 21]))

sắp xếp: [1, 2, 4, 5] | pre: [1, 3, 7, 12]
  q=3: lo=0 hi=4 mid=2 pre[mid]=7 -> pre[mid] > q, hi = mid
  q=3: lo=0 hi=2 mid=1 pre[mid]=3 -> pre[mid] <= q, lo = mid+1
q=3 -> trả lời 2
  q=10: lo=0 hi=4 mid=2 pre[mid]=7 -> pre[mid] <= q, lo = mid+1
  q=10: lo=3 hi=4 mid=3 pre[mid]=12 -> pre[mid] > q, hi = mid
q=10 -> trả lời 3
  q=21: lo=0 hi=4 mid=2 pre[mid]=7 -> pre[mid] <= q, lo = mid+1
  q=21: lo=3 hi=4 mid=3 pre[mid]=12 -> pre[mid] <= q, lo = mid+1
q=21 -> trả lời 4
[2, 3, 4]


## 9. Code nộp LeetCode

In [13]:
from typing import List
from bisect import bisect_right
from itertools import accumulate

class Solution:
    def answerQueries(self, nums: List[int], queries: List[int]) -> List[int]:
        pre = list(accumulate(sorted(nums)))
        return [bisect_right(pre, q) for q in queries]

print(Solution().answerQueries([4, 5, 2, 1], [3, 10, 21]), Solution().answerQueries([2, 3, 4, 5], [1]))

[2, 3, 4] [0]


## 10. Vì sao thuật toán đúng

- **Greedy đúng:** giả sử một phương án tối ưu dùng `k` phần tử có tổng `<= q`. Thay chúng bằng `k` phần tử **nhỏ nhất** của `nums` thì tổng không tăng (tổng của `k` số nhỏ nhất luôn ≤ tổng của bất kỳ `k` số nào), vẫn `<= q`. Vậy tối ưu có thể giả định là `k` số nhỏ nhất.
- **Thứ tự không quan trọng:** mọi tập chỉ số đều tạo được một dãy con (đọc theo chỉ số tăng dần), nên ta tự do chọn bất kỳ tập nào.
- **Tính đơn điệu:** `pre` tăng ngặt vì phần tử `>= 1`. Nếu `pre[j] <= q` thì mọi `pre[i]` với `i < j` cũng `<= q`; do đó các `j` hợp lệ là một tiền tố → có thể nhị phân và `bisect_right` trả số phần tử đó.
- **Hai con trỏ:** vì truy vấn đã sắp xếp tăng, tập `j` hợp lệ chỉ mở rộng, nên không cần lùi.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `m = len(queries)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute mô phỏng | `O(n log n + m·n)` | `O(n + m)` |
| 2. Prefix + quét | `O(n log n + m·n)` | `O(n + m)` |
| 3. Prefix + `bisect_right` | `O(n log n + m log n)` | `O(n + m)` |
| 4. Hai con trỏ offline | `O(n log n + m log m)` | `O(n + m)` |

Sắp xếp là phần tốn nhất ở cách 3 và 4. Không có khác biệt average/worst đáng kể (Timsort worst `O(n log n)`).

## 12. Bộ test

Test 1: ví dụ của đề và các edge case. Đối chiếu tất cả cài đặt.

In [14]:
impls = [SolutionBrute(), SolutionScan(), SolutionBisect(), SolutionOffline(), Solution()]
tests = [
    ([4, 5, 2, 1], [3, 10, 21], [2, 3, 4]),
    ([2, 3, 4, 5], [1], [0]),
    ([1], [1], [1]),                       # n = 1, vừa đủ (bắt lỗi bisect_left)
    ([5], [4], [0]),
    ([1, 1, 1, 1], [1, 2, 3, 4, 5], [1, 2, 3, 4, 4]),
    ([3 * 10**5] * 3, [3 * 10**5, 6 * 10**5, 9 * 10**5], [1, 2, 3]),
    ([3, 1, 2], [3, 3, 6], [2, 2, 3]),     # truy vấn lặp lại, không có thứ tự
    ([9, 8, 7], [24, 23, 7], [3, 2, 1]),   # tổng đúng bằng q ở biên
]
for nums, qs, exp in tests:
    for im in impls:
        got = im.answerQueries(list(nums), list(qs))
        assert got == exp, (type(im).__name__, nums, qs, got, exp)
print("tất cả test cố định OK")

tất cả test cố định OK


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n, m <= 1000`, giá trị trong `1..10^6`), đối chiếu thêm với brute force cực kỳ ngây thơ: thử mọi `k` và so tổng `k` số nhỏ nhất.

In [15]:
import random
random.seed(2389)

def oracle(nums, q):
    s = sorted(nums)
    best = 0
    for k in range(len(s) + 1):
        if sum(s[:k]) <= q:
            best = k
    return best

for _ in range(300):
    n = random.randint(1, 30)
    m = random.randint(1, 30)
    hi = random.choice([10, 100, 10**6])
    nums = [random.randint(1, hi) for _ in range(n)]
    qs = [random.randint(1, min(10**6, hi * n)) for _ in range(m)]
    exp = [oracle(nums, q) for q in qs]
    for im in impls:
        assert im.answerQueries(list(nums), list(qs)) == exp, (type(im).__name__, nums, qs)
print("300 test ngẫu nhiên nhỏ OK")

# test lớn đúng giới hạn: n = m = 1000
nums = [random.randint(1, 10**6) for _ in range(1000)]
qs = [random.randint(1, 10**6) for _ in range(1000)]
res = [im.answerQueries(list(nums), list(qs)) for im in impls]
assert all(r == res[0] for r in res)
print("test n = m = 1000 OK, đáp án đầu:", res[0][:5])

300 test ngẫu nhiên nhỏ OK
test n = m = 1000 OK, đáp án đầu: [5, 30, 39, 28, 33]


Test 3: kiểm tra tính chất — `answer` không giảm khi `q` tăng, và hàm không làm thay đổi `nums`/`queries`.

In [16]:
random.seed(7)
nums = [random.randint(1, 50) for _ in range(40)]
qs = sorted(random.randint(1, 2000) for _ in range(60))
ans = Solution().answerQueries(nums, qs)
assert ans == sorted(ans), "đáp án phải không giảm theo q"
copy_n, copy_q = list(nums), list(qs)
Solution().answerQueries(nums, qs)
assert nums == copy_n and qs == copy_q
print("tính đơn điệu và không đổi input: OK")

tính đơn điệu và không đổi input: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên sắp xếp trước khi cộng dồn

In [17]:
from itertools import accumulate
from bisect import bisect_right

def wrong_unsorted(nums, queries):
    pre = list(accumulate(nums))           # SAI: chưa sort, pre không tối ưu
    return [bisect_right(pre, q) for q in queries]

nums, qs = [4, 5, 2, 1], [3, 10, 21]
print("sai :", wrong_unsorted(nums, qs))
print("đúng:", Solution().answerQueries(nums, qs))
assert wrong_unsorted(nums, qs) != Solution().answerQueries(nums, qs)

sai : [0, 2, 4]
đúng: [2, 3, 4]


Vì sao sai: với `q = 3`, mảng chưa sắp xếp có `pre = [4, 9, 11, 12]`, không tiền tố nào `<= 3` → `0`, trong khi chọn `[2, 1]` được `2`. **Sửa:** `sorted(nums)` trước khi `accumulate`.

### Lỗi 2: dùng `bisect_left` khi tổng bằng đúng `q`

In [18]:
from bisect import bisect_left

def wrong_left(nums, queries):
    pre = list(accumulate(sorted(nums)))
    return [bisect_left(pre, q) for q in queries]   # SAI: loại tiền tố có tổng == q

print("sai :", wrong_left(nums, qs))
print("đúng:", Solution().answerQueries(nums, qs))
assert wrong_left(nums, qs) != Solution().answerQueries(nums, qs)

sai : [1, 3, 4]
đúng: [2, 3, 4]


Vì sao sai: `bisect_left` đếm số phần tử **nhỏ hơn** `q`, bỏ qua tiền tố có tổng **bằng** `q` (đề cho phép `<=`). Với `q = 3`, `pre = [1, 3, 7, 12]` → `bisect_left` cho `1`, còn đúng là `2`. **Sửa:** `bisect_right`.

### Lỗi 3: trả về chỉ số thay vì độ dài / tự trừ 1

In [19]:
def wrong_index(nums, queries):
    pre = list(accumulate(sorted(nums)))
    return [bisect_right(pre, q) - 1 for q in queries]   # SAI: trừ 1 thừa

print("sai :", wrong_index(nums, qs))
print("đúng:", Solution().answerQueries(nums, qs))
assert wrong_index(nums, qs) != Solution().answerQueries(nums, qs)

sai : [1, 2, 3]
đúng: [2, 3, 4]


Vì sao sai: `bisect_right` đã trả đúng **số lượng** phần tử; trừ 1 biến nó thành chỉ số của phần tử cuối và lệch 1 đơn vị (còn có thể ra `-1`).

### Lỗi 4 (bonus): sắp xếp truy vấn rồi quên khôi phục thứ tự

Ở cách offline, nếu viết `queries.sort()` rồi trả `answer` theo thứ tự đã đổi thì đáp án không khớp với truy vấn gốc. Luôn lưu **chỉ số gốc** (`answer[i] = ...`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn nhiều phần tử nhất sao cho **tổng ≤ giới hạn**" + thứ tự không quan trọng → **sort + greedy lấy nhỏ nhất**.
- Nhiều truy vấn cùng một mảng → tính trước **prefix sum**, mỗi truy vấn nhị phân.
- Nhớ phân biệt: "tổng `<= q`" dùng `bisect_right`, "tổng `< q`" dùng `bisect_left`.
- Mảng prefix chỉ nhị phân được khi **đơn điệu**; ở đây nhờ `nums[i] >= 1`. Nếu có số `<= 0` phải kiểm tra lại.
- Nếu `m` rất lớn, nghĩ tới xử lý offline (sắp xếp truy vấn + hai con trỏ).
- Đừng đưa `list.sort()` vào biểu thức: nó trả `None`.

## 15. Biến thể của bài

1. Trả về **tổng lớn nhất không vượt `q`** của một dãy con có độ dài ≤ k (bài knapsack nhẹ).
2. Dãy con phải **gồm đúng các phần tử liên tiếp** (subarray) — lúc đó cần sliding window / hai con trỏ vì không còn tự do chọn.
3. "Tối đa số bài giải được với tổng thời gian ≤ T" (cùng ý tưởng mua đồ).
4. Truy vấn tăng/giảm động (cần cấu trúc như cây Fenwick).
5. Lấy **ít** phần tử nhất để tổng **đạt tối thiểu `q`** (sắp xếp giảm dần rồi cộng dồn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute mô phỏng | cộng từng số nhỏ cho mỗi truy vấn | `O(n log n + m·n)` | `O(n + m)` | học greedy, `n·m` nhỏ |
| 2. Prefix + quét | tiền tố một lần, đếm tuyến tính | `O(n log n + m·n)` | `O(n + m)` | chuyển tiếp sang nhị phân |
| 3. Prefix + `bisect_right` | nhị phân trên tiền tố | `O(n log n + m log n)` | `O(n + m)` | **chuẩn**, ngắn gọn, nhanh |
| 4. Hai con trỏ offline | sort truy vấn, `j` chỉ tiến | `O(n log n + m log m)` | `O(n + m)` | rất nhiều truy vấn, ngôn ngữ không có bisect |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `nums = [3, 3, 3]`, `queries = [2, 3, 6, 9, 10]` → kết quả là gì?

**Bài 2 (tính tay):** `nums = [7, 1, 5]`, `queries = [1, 6, 8, 13]`; viết `pre` và đáp án.

**Bài 3 (điền TODO):** hoàn thành `count_leq(pre, q)` đếm số phần tử `<= q` bằng `bisect_right`.

**Bài 4 (tự viết lại):** viết `max_sum_leq(nums, q)` trả về **tổng lớn nhất** có thể của tiền tố nhỏ-nhất không vượt `q` (tức `pre[answer-1]`, hoặc `0` nếu không chọn gì).

**Bài 5 (mini challenge):** viết `min_items_to_reach(nums, q)` — số phần tử **ít nhất** để tổng `>= q` (chọn số lớn trước); trả `-1` nếu tổng cả mảng `< q`.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().answerQueries([3, 3, 3], [2, 3, 6, 9, 10]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_leq(pre, q):
    # TODO: dùng bisect_right để đếm số phần tử <= q
    pass

[0, 1, 2, 3, 3]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** luôn sắp xếp tăng rồi cộng dồn. Đáp án là số tiền tố có tổng không vượt `q`.

**Mức 2 (vừa):** Bài 3: `return bisect_right(pre, q)`. Bài 4: `k = count_leq(pre, q)`, rồi `pre[k - 1] if k else 0`. Bài 5: sắp xếp **giảm dần**, cộng dồn, trả về chỉ số đầu tiên mà tổng `>= q`.

**Mức 3 (gần đáp án):** Bài 5: duyệt `for i, x in enumerate(sorted(nums, reverse=True), 1): total += x; if total >= q: return i`; sau vòng lặp trả `-1`. Bài 2: `sorted = [1,5,7]`, `pre = [1,6,13]` → `[1,2,2,3]`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
from bisect import bisect_right
from itertools import accumulate

# Bài 1: [3,3,3], pre = [3,6,9]
assert Solution().answerQueries([3, 3, 3], [2, 3, 6, 9, 10]) == [0, 1, 2, 3, 3]

# Bài 2: sorted [1,5,7], pre [1,6,13]
assert list(accumulate(sorted([7, 1, 5]))) == [1, 6, 13]
assert Solution().answerQueries([7, 1, 5], [1, 6, 8, 13]) == [1, 2, 2, 3]

# Bài 3
def count_leq(pre, q):
    return bisect_right(pre, q)
assert count_leq([1, 3, 7, 12], 3) == 2
assert count_leq([1, 3, 7, 12], 0) == 0

# Bài 4
def max_sum_leq(nums, q):
    pre = list(accumulate(sorted(nums)))
    k = count_leq(pre, q)
    return pre[k - 1] if k else 0
assert max_sum_leq([4, 5, 2, 1], 10) == 7
assert max_sum_leq([4, 5, 2, 1], 0) == 0
assert max_sum_leq([4, 5, 2, 1], 100) == 12

# Bài 5
def min_items_to_reach(nums, q):
    total = 0
    for i, x in enumerate(sorted(nums, reverse=True), 1):
        total += x
        if total >= q:
            return i
    return -1
assert min_items_to_reach([4, 5, 2, 1], 9) == 2
assert min_items_to_reach([4, 5, 2, 1], 13) == -1
assert min_items_to_reach([4, 5, 2, 1], 1) == 1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sort + prefix + bisect" cho truy vấn `tổng <= q`:**

```python
from bisect import bisect_right
from itertools import accumulate

pre = list(accumulate(sorted(nums)))       # tăng dần
answer = [bisect_right(pre, q) for q in queries]
```

**Checklist trước khi submit:**

- [ ] Đã `sorted(nums)` **trước** khi `accumulate`?
- [ ] Dùng `bisect_right` (tổng bằng `q` vẫn được tính)?
- [ ] Không trừ/cộng thêm 1 vào kết quả `bisect_right`?
- [ ] Đáp án theo **đúng thứ tự** của `queries`?
- [ ] Đã thử `q` nhỏ hơn mọi phần tử (kết quả 0) và `q` lớn hơn tổng (kết quả `n`)?
- [ ] Đã đối chiếu với brute force bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `answerQueries(self, nums: List[int], queries: List[int]) -> List[int]`?